In [57]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.tuning import ParamGridBuilder
from pyspark.ml import Pipeline, PipelineModel
import matplotlib.pyplot as plt
import numpy as np

In [17]:
spark = SparkSession.builder \
    .appName("Segmentacion_Perfilado_Clientes") \
    .getOrCreate()

clientes_df = spark.read.parquet("/home/jovyan/work/data/CLIENTS", header=True, inferSchema=True)
behavioural_df = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL", header=True, inferSchema=True)

In [ ]:
# df_id_unido = clientes_df.alias("c").join(
#     behavioural_df.alias("b"),
#     on="CLIENT_ID",
#     how="inner"
# )

In [ ]:
# df_id_unido.show(1)

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----------------

In [ ]:
# len(df_id_unido.columns)

58

In [ ]:
# NUMERICAL_COLS = [
#     "JOB_SENIORITY",
#     "INSTALLMENT",
#     "FAMILY_SIZE",
#     "PROACTIVE_SCORING",
#     "BEHAVIORAL_SCORING",
#     "DAYS_LAST_INFO_CHANGE",
#     "NUMBER_OF_PRODUCTS",
#     "TOTAL_INCOME",
#     "AMOUNT_PRODUCT",
#     "INSTALLMENT",
#     "REGION_SCORE",
# ]

# CATEGORICAL_COLS = [
# ]

In [ ]:
# df_imputado = df_id_unido.na.fill(0, NUMERICAL_COLS)
# df_imputado = df_imputado.na.fill("NA_MISSING", subset=CATEGORICAL_COLS)

In [ ]:
# df_imputado.show(5)

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----------------

In [ ]:
# #Todas las columnas ya están limpias y sin valores nulos

# all_columns = NUMERICAL_COLS + CATEGORICAL_COLS

In [ ]:
# indexers = [
#     StringIndexer(inputCol=c, outputCol=c + "_Index", handleInvalid="keep")
#     for c in CATEGORICAL_COLS
# ]

# encoders = [
#     OneHotEncoder(inputCols=[c + "_Index"], outputCols=[c + "_OHE"], dropLast=True)
#     for c in CATEGORICAL_COLS
# ]

In [ ]:
# indexers

[]

In [58]:
# assembler = VectorAssembler(
#     inputCols=all_columns,
#     outputCol="unscaled_features" # Nombre temporal antes de escalar
# )


# scaler = StandardScaler(
#     inputCol="unscaled_features",
#     outputCol="features", # ESTA es tu columna final
#     withStd=True,
#     withMean=False # No centrar para datos dispersos (OHE), si tienes muchos zeros
#)

In [29]:
# PCA_COMPONENTS = 10 
# pca = PCA(k=PCA_COMPONENTS, inputCol="features", outputCol="pca_features")

In [ ]:
# pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, pca])

# pipeline_model = pipeline.fit(df_imputado)

# df_features = pipeline_model.transform(df_imputado)

# df_features.select("CLIENT_ID", "features").show(5, False)

# print(f"Dimensiones totales del vector 'features': {len(all_columns)}")

+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|CLIENT_ID   |features                                                                                                                                                                                                         |
+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ES182147947X|[0.29233387252426585,2.614051124979187,2.281310270135329,3.8095503356063487,0.0,2.5873136648080646,0.0,1.6056954949819635,2.7444960406161614,2.614051124979187,2.7595859054723357]                               |
|ES182389511E|[0.05498566749779831,1.0976702460102052,2.281310270135329,3.735953097072094,1.99944595

In [63]:
columnas_beh = [
    "CREDICT_CARD_BALANCE",
    "CREDIT_CARD_LIMIT",
    "CREDIT_CARD_DRAWINGS_ATM",
    "CREDIT_CARD_DRAWINGS_POS",
    "CREDIT_CARD_DRAWINGS_OTHER",
    "CREDIT_CARD_DRAWINGS",
    "CREDIT_CARD_PAYMENT",
    "NUMBER_DRAWINGS_ATM",
    "NUMBER_DRAWINGS",
    "NUMBER_INSTALMENTS"
]

columnas_base = ["CLIENT_ID"] + columnas_beh

In [ ]:
#  comprobación de que las columnas funcionan
#  for col in columnas_beh:
#     behavioural_df.select(col).distinct().show(5)

In [45]:
assembler = VectorAssembler(
    inputCols=columnas_beh,
    outputCol="features"
)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=False
)

In [65]:
feature_pipeline = Pipeline(stages=[assembler, scaler])

behavioural_df_num = behavioural_df.select(columnas_base)

df_features = feature_pipeline.fit(behavioural_df_num).transform(behavioural_df_num)

# VAMOS CON EL FOKIN K-MEANS

In [66]:
FEATURES_COL = "scaled_features" 
K_range = np.arange(2, 11).tolist()
random_seed = 777

inertias = []
silhouettes = []

# Definir el estimador K-Means
kmeans = KMeans(featuresCol=FEATURES_COL, seed=random_seed)

# Definir el evaluador de Silhouette (métrica de calidad de clustering)
evaluator = ClusteringEvaluator(
    featuresCol=FEATURES_COL, 
    predictionCol="prediction", 
    metricName="silhouette", 
    distanceMeasure="squaredEuclidean"
)

# Definir la grilla de parámetros
paramGrid = (ParamGridBuilder()
    .addGrid(kmeans.k, K_range)
    .build()
)

print(f"Iniciando Grid Search distribuido sobre K: {K_range}...")

# --- 2. BÚSQUEDA DISTRIBUIDA ---
for params in paramGrid:
    # Entrenar el modelo con el K actual
    model = kmeans.fit(df_features, params)
    
    # Calcular WCSS (Inercia)
    wcss = model.summary.trainingCost
    inertias.append(wcss)
    
    # Transformar y calcular Silhouette Score
    df_predictions = model.transform(df_features)
    sil_score = evaluator.evaluate(df_predictions)
    silhouettes.append(sil_score)
    
    k = params.get(kmeans.k)
    print(f"K={k}: WCSS (Inercia) = {wcss:.2f}, Silhouette Score = {sil_score:.4f}")

# --- 3. GUARDAR RESULTADOS ---
# Crear un DataFrame con los resultados
df_results = spark.createDataFrame(
    zip(K_range, inertias, silhouettes), 
    ["k", "Inertia", "Silhouette"]
)

# Guardar los resultados en un CSV pequeño (¡ahora sí se puede hacer .collect()!)
df_results.toPandas().to_csv("kmeans_metrics_results.csv", index=False)
print("\n✅ Cálculo distribuido completado.")
print("El resultado para plotear se ha guardado en: 'kmeans_metrics_results.csv'")

Iniciando Grid Search distribuido sobre K: [2, 3, 4, 5, 6, 7, 8, 9, 10]...
K=2: WCSS (Inercia) = 16002389.95, Silhouette Score = 0.9888
K=3: WCSS (Inercia) = 12265638.47, Silhouette Score = 0.8935
K=4: WCSS (Inercia) = 10837008.45, Silhouette Score = 0.6559
K=5: WCSS (Inercia) = 10251290.81, Silhouette Score = 0.6542
K=6: WCSS (Inercia) = 9511185.67, Silhouette Score = 0.5106
K=7: WCSS (Inercia) = 7822350.71, Silhouette Score = 0.4492
K=8: WCSS (Inercia) = 7790130.16, Silhouette Score = 0.5045
K=9: WCSS (Inercia) = 6604418.46, Silhouette Score = 0.4775
K=10: WCSS (Inercia) = 6392096.12, Silhouette Score = 0.5129

✅ Cálculo distribuido completado.
El resultado para plotear se ha guardado en: 'kmeans_metrics_results.csv'


In [71]:
# ASUMIMOS que df_features ahora contiene CLIENT_ID y scaled_features

K_OPTIMO = 2 

kmeans_final = KMeans(
    featuresCol="scaled_features", 
    predictionCol="segmento_cliente", 
    k=K_OPTIMO,
    seed=42
)
model_final = kmeans_final.fit(df_features)

# La transformación simplemente añade la columna 'segmento_cliente' a df_features
df_segmentado = model_final.transform(df_features)

# Eliminamos la línea incorrecta de withColumn
# df_segmentado = df_segmentado.withColumn(...) <--- ELIMINAR ESTO

print(f"Modelo K-Means entrenado con K={K_OPTIMO}. Segmentos asignados.")
# La selección ahora funcionará porque CLIENT_ID está en el DataFrame desde el inicio
df_segmentado.select("CLIENT_ID", "segmento_cliente").show(5)


Modelo K-Means entrenado con K=2. Segmentos asignados.
+------------+----------------+
|   CLIENT_ID|segmento_cliente|
+------------+----------------+
|ES182147947X|               0|
|ES182389511E|               0|
|ES182423955L|               0|
|ES182265271Q|               0|
|ES182332566J|               0|
+------------+----------------+
only showing top 5 rows



In [76]:
# 1. Calcular el promedio de las variables clave por segmento
# Agrupamos por el resultado del clustering ('segmento_cliente')
df_perfil_numerico = df_segmentado.groupBy("segmento_cliente").agg(
    *[F.mean(col).alias(f"MEAN_{col}") for col in columnas_beh]
).orderBy("segmento_cliente")

# 2. Mostrar la tabla de perfiles
# Esto te permitirá comparar las medias de los dos segmentos (0 y 1)
df_perfil_numerico.show(truncate=False)

# 3. (Opcional) Guardar en CSV para un análisis más detallado o Excel
df_perfil_numerico.toPandas().to_csv("segment_profiles_quick_view.csv", index=False)

+----------------+-------------------------+----------------------+-----------------------------+-----------------------------+-------------------------------+-------------------------+------------------------+------------------------+--------------------+-----------------------+
|segmento_cliente|MEAN_CREDICT_CARD_BALANCE|MEAN_CREDIT_CARD_LIMIT|MEAN_CREDIT_CARD_DRAWINGS_ATM|MEAN_CREDIT_CARD_DRAWINGS_POS|MEAN_CREDIT_CARD_DRAWINGS_OTHER|MEAN_CREDIT_CARD_DRAWINGS|MEAN_CREDIT_CARD_PAYMENT|MEAN_NUMBER_DRAWINGS_ATM|MEAN_NUMBER_DRAWINGS|MEAN_NUMBER_INSTALMENTS|
+----------------+-------------------------+----------------------+-----------------------------+-----------------------------+-------------------------------+-------------------------+------------------------+------------------------+--------------------+-----------------------+
|0               |623.2155948311471        |1717.085803437802     |16.63747472907447            |8.856158468354822            |0.9824616722535193            